# 00 · Hiểu cấu trúc và xác minh nguồn PTB-XL
**Câu hỏi:** ta đang nhận dữ liệu gì, một con số trong file có đơn vị gì, và dựa vào đâu biết nguồn không bị thiếu/sai?

**Đầu vào:** PTB-XL v1.0.3 trên PhysioNet hoặc bản sao đã có. **Đầu ra:** nguồn 500 Hz được đối chiếu SHA256 và `source_verification.json`. **Tiếp theo:** 01 tạo nhãn/chia tập.

Pipeline dùng 12 chuyển đạo, 10 giây, nguồn `records500` (5.000 điểm/chuyển đạo). Đây là waveform ECG, không phải ảnh ECG. Nguồn có cả bản 100 Hz; tăng mẫu bản 100 Hz không khôi phục thông tin của bản 500 Hz. Chỉ tải thành phần 500 Hz cần dùng khoảng 2,65 GB; release float32 sẽ lớn hơn vì cách lưu khác.

Bài toán hiện tại là 5 nhóm chẩn đoán ECG đa nhãn; chưa có nhãn suy tim.

**Trạng thái bản sửa 01/11/2026:** source giữ sạch output; các hình/bảng chỉ có sau khi Line chạy trên máy đích. Output lưu ngày 24/10/2026 thuộc phiên bản notebook cũ, không chứng minh các cell mới đã chạy. Các cell ghi “TOÀN BỘ” có thể đọc/ghi nhiều GB; không chạy chỉ để xem nội dung.

## 1. Đường dẫn và hợp đồng lần chạy
Mỗi notebook chạy trong kernel mới. Đường dẫn có thể đổi qua biến môi trường; `TARGET_FS`, `PREPROCESSING` và `EXPECTED_RECORDS` phải nhất quán giữa các notebook. Giữ 500 Hz/minimal cho bản chuẩn hiện tại. Thử cấu hình khác phải dùng output mới; không sửa dữ liệu đã audit.

In [ ]:
from pathlib import Path
import os
import sys

def tim_thu_muc_project():
    explicit = os.environ.get("PTBXL_PROJECT_ROOT")
    candidates = [Path(explicit).expanduser().resolve()] if explicit else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "ptbxl_pipeline").is_dir() and (candidate / "requirements-ptbxl.txt").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Không tìm thấy project. Đặt PTBXL_PROJECT_ROOT tới thư mục chứa ptbxl_pipeline và requirements-ptbxl.txt.")

ROOT = tim_thu_muc_project()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW_DIR = Path(os.environ.get("PTBXL_RAW_DIR", str(ROOT / "data/raw/ptbxl/1.0.3"))).expanduser().resolve()
CACHE_DIR = ROOT / "data/cache"
OUTPUT_DIR = Path(os.environ.get("PTBXL_OUTPUT_DIR", str(ROOT / "data/processed/ptbxl_v1.0.3_500hz_minimal"))).expanduser().resolve()
REPORT_DIR = ROOT / "artifacts/ptbxl_reports" / OUTPUT_DIR.name
TARGET_FS = 500
PREPROCESSING = "minimal"
EXPECTED_RECORDS = 21799
print("Project:", ROOT)
print("Dữ liệu gốc:", RAW_DIR)
print("Bộ xuất:", OUTPUT_DIR)

## 2. TOÀN BỘ: lấy nguồn và kiểm tra checksum
`ensure_source` lấy file thiếu từ mirror công khai do PhysioNet công bố, hoặc dùng archive chính thức đã có. Nó đối chiếu metadata và các cặp `.hea/.dat` bằng SHA256. Nếu chỉ dùng file có sẵn, đổi `download=False`; thiếu file khi đó là lỗi rõ ràng.

Checksum xác nhận nội dung khớp bản phát hành, không xác nhận nhãn đúng về mặt lâm sàng. Cell này có thể tải khoảng 2,65 GB và kiểm tra toàn bộ nguồn; chỉ chạy khi đã sẵn sàng trên máy xử lý.

In [ ]:
import json
from IPython.display import JSON, display
from ptbxl_pipeline.acquire import ensure_source

source_report = ensure_source(RAW_DIR, CACHE_DIR, download=True)
assert source_report.get("verified") is True
assert source_report.get("source_version") == "1.0.3"
source_summary = {key: value for key, value in source_report.items() if key != "file_hashes"}
source_summary["so_file_da_doi_chieu_SHA256"] = len(source_report["file_hashes"])
display(JSON(source_summary))
verification_path = RAW_DIR / "source_verification.json"
assert verification_path.is_file(), "Chưa có biên bản xác minh nguồn."
print("Biên bản:", verification_path)

## 3. Từ điển dữ liệu: giữ vai trò từng cột
Không đưa mọi cột vào model. `report`, mã SCP và các trường xác nhận liên quan nhãn có thể làm rò rỉ mục tiêu nếu dùng làm đầu vào cho bài toán chỉ dùng waveform. Định danh dùng để truy vết/chia tập, không phải đặc trưng bệnh.

In [ ]:
import pandas as pd
from IPython.display import display

database = pd.read_csv(RAW_DIR / "ptbxl_database.csv", keep_default_na=False, low_memory=False)
statements = pd.read_csv(RAW_DIR / "scp_statements.csv", index_col=0)
assert len(database) == EXPECTED_RECORDS
assert database["ecg_id"].is_unique
assert {"patient_id", "filename_hr", "scp_codes", "strat_fold"}.issubset(database)
field_guide = {
    "ecg_id": "ID của ECG, khóa nối dữ liệu",
    "patient_id": "ID bệnh nhân: kiểm tra không giao bệnh nhân giữa các tập",
    "strat_fold": "Fold chính thức: 1–8 train; 9 validation; 10 test",
    "filename_hr": "Đường dẫn cặp WFDB 500 Hz, không có phần mở rộng",
    "filename_lr": "Đường dẫn bản 100 Hz; pipeline này không dùng",
    "scp_codes": "Mã SCP và likelihood; dùng làm nguồn nhãn",
    "report": "Báo cáo gốc; không phải đầu vào của baseline waveform",
    "age": "Tuổi có ẩn danh ở nhóm cao tuổi; không xem mã 300+ là tuổi thật",
    "sex": "Giới tính theo mã nguồn; giữ thiếu/không biết riêng",
    "height": "Chiều cao; có thiếu, không tự thay bằng 0",
    "weight": "Cân nặng; có thiếu, không tự thay bằng 0",
    "device": "Thiết bị: có thể dùng để kiểm tra subgroup/domain shift sau này",
    "baseline_drift": "Chú thích trôi đường nền; trống không chứng minh sạch",
    "static_noise": "Chú thích nhiễu tĩnh; không phải phép đo SNR",
    "burst_noise": "Chú thích nhiễu từng đợt",
    "electrodes_problems": "Chú thích vấn đề điện cực",
}
display(pd.DataFrame([{"cot": name, "vai_tro": role, "co_trong_nguon": name in database} for name, role in field_guide.items()]))
display(pd.DataFrame({"ECG": [len(database)], "benh_nhan": [database.patient_id.nunique()], "ma_SCP": [len(statements)]}))

## 4. Theo một ECG train từ metadata tới header
Chọn ECG train có ID nhỏ nhất một cách xác định. Đây là ví dụ đọc định dạng, không phải mẫu đại diện cho mọi bệnh. `.hea` mô tả số lead, tần số, số mẫu, đơn vị, gain và baseline; `.dat` chứa các giá trị số hóa. Không suy đơn vị từ biên độ nhìn thấy.

In [ ]:
import wfdb
from ptbxl_pipeline.core import LEADS

source_train = database.loc[pd.to_numeric(database.strat_fold).between(1, 8)].sort_values("ecg_id")
example = source_train.iloc[0]
record_path = RAW_DIR / example.filename_hr
header = wfdb.rdheader(str(record_path))
assert header.fs == 500 and header.sig_len == 5000 and header.n_sig == 12
display(example[["ecg_id", "patient_id", "strat_fold", "filename_hr"]].to_frame("gia_tri"))
display(pd.DataFrame({"lead_nguon": header.sig_name, "don_vi_header": header.units, "adc_gain": header.adc_gain, "baseline": header.baseline, "format": header.fmt}))
print("Thời lượng (s):", header.sig_len / header.fs)
print("Thứ tự lead đầu ra:", LEADS)

## 5. Số ADC khác điện áp mV như thế nào?
WFDB dùng quan hệ `physical = (digital − baseline) / adc_gain` trong đơn vị ghi ở header. Khi đã gọi `physical=True`, `p_signal` đã được giải mã; chia gain lần nữa sẽ sai biên độ. Hàm minh họa dưới đây chỉ sắp lead và đổi đơn vị sang mV, không thay thế pipeline QC.

Thử phép tính trên cùng một ECG giúp phát hiện nhầm digital/physical, nhầm trục hoặc nhầm đơn vị trước khi xuất hàng chục nghìn file.

In [ ]:
import numpy as np

def physical_mV(record, leads):
    """Minh họa đọc header: trả [lead,time] mV, chưa áp dụng QC loại lead hằng."""
    names = [str(name).strip().upper() for name in record.sig_name]
    wanted = [str(name).upper() for name in leads]
    if len(names) != len(set(names)) or not set(wanted).issubset(names):
        raise ValueError("missing/duplicate lead trong header")
    order = [names.index(name) for name in wanted]
    factors = {"mv": 1.0, "uv": 0.001, "µv": 0.001, "μv": 0.001, "v": 1000.0}
    units = [str(record.units[i]).strip().lower() for i in order]
    if any(unit not in factors for unit in units):
        raise ValueError("unsupported unit trong header")
    values = np.asarray(record.p_signal, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != len(names):
        raise ValueError("shape không khớp header")
    return (values[:, order] * np.asarray([factors[unit] for unit in units])).T

In [ ]:
physical_record = wfdb.rdrecord(str(record_path), physical=True)
digital_record = wfdb.rdrecord(str(record_path), physical=False)
manual_physical = (digital_record.d_signal.astype(np.float64) - np.asarray(header.baseline)) / np.asarray(header.adc_gain)
assert np.isfinite(manual_physical).all()
np.testing.assert_allclose(manual_physical, physical_record.p_signal, rtol=1e-10, atol=1e-10)
wave_mV = physical_mV(physical_record, LEADS)
assert wave_mV.shape == (12, 5000)
display(pd.DataFrame({"lead": LEADS, "min_mV": wave_mV.min(axis=1), "max_mV": wave_mV.max(axis=1), "std_mV": wave_mV.std(axis=1)}))
print("Sai số lớn nhất giải mã ADC:", float(np.max(np.abs(manual_physical - physical_record.p_signal))))

## 6. Nhìn waveform đúng trục thời gian và đơn vị
Hình chỉ dùng để kiểm tra đọc file. Không gán bệnh từ biểu đồ này, và không đổi ngưỡng QC vì một mẫu trông khác thường. Các biến thiên bệnh lý có thể là thông tin cần giữ.

In [ ]:
import matplotlib.pyplot as plt

time_seconds = np.arange(wave_mV.shape[1]) / header.fs
fig, axes = plt.subplots(6, 2, figsize=(15, 12), sharex=True)
for index, axis in enumerate(axes.flat):
    axis.plot(time_seconds, wave_mV[index], linewidth=0.65)
    axis.set_title(LEADS[index])
    axis.set_ylabel("mV")
    axis.grid(alpha=0.2)
for axis in axes[-1]:
    axis.set_xlabel("Giây")
fig.suptitle(f"Nguồn 500 Hz — ECG train {int(example.ecg_id)}; một ví dụ định dạng")
fig.tight_layout()
plt.show()

## Quyết định và giới hạn
- Đi tiếp khi source count/ID và checksum đạt, header có 12 lead/500 Hz/10 giây và phép giải mã ADC khớp.
- Nếu thiếu file hoặc sai checksum, giải quyết ở nguồn; không bỏ kiểm tra hoặc đổi version cho qua.
- Nguồn đúng định dạng vẫn có thể nhiễu, thiếu nhãn hoặc chứa chú thích không nhất quán. Notebook 01 phân tích nhãn; notebook 02 phân tích kỹ thuật tín hiệu.
- Không có kết quả hiệu năng model từ bước này.

Nguồn đối chiếu 01/11/2026: [PTB-XL 1.0.3](https://physionet.org/content/ptb-xl/1.0.3/), [WFDB I/O](https://wfdb.readthedocs.io/en/latest/io.html). Giữ giấy phép CC BY 4.0 và trích dẫn dữ liệu khi sử dụng. Tiếp theo: **01_tao_nhan_va_chia_tap.ipynb**.